# Bab 5: Mempelajari Banyak Bobot Sekaligus: Generalisasi Gradient Descent
*(Learning Multiple Weights at a Time: Generalizing Gradient Descent)*

## Eksplorasi Teoritis Mendalam (Theoretical Deep-Dive)

Pada Bab 4, kita telah berhasil membuat jaringan saraf belajar mengoreksi satu bobot berdasarkan satu masukan (input). Namun, di dunia nyata, prediksi selalu bergantung pada banyak titik data sekaligus (seperti yang kita pelajari di Bab 3). Bab ini menggabungkan konsep dari Bab 3 dan Bab 4: bagaimana kita melakukan *Gradient Descent* ketika ada **banyak masukan dan banyak bobot** yang memengaruhi satu prediksi.

### 1. Masalah Pembagian Kesalahan (Credit Assignment Problem)
Ketika sebuah jaringan saraf memiliki 3 input dan menghasilkan 1 tebakan, lalu ternyata tebakan tersebut salah, pertanyaan utamanya adalah: **Input mana yang paling bertanggung jawab atas kesalahan tersebut?**
Algoritma harus bisa mendistribusikan "hukuman" (error) secara adil ke masing-masing bobot. Proses ini disebut sebagai *Credit Assignment*.

### 2. Aturan Turunan Jaringan Saraf (The Learning Rule)
Untuk mendistribusikan *error* ke banyak bobot, kita masih menggunakan rumus dasar yang sama dari Bab 4:
`weight_delta = input * delta`

*   **Mekanisme Proporsional:** Bobot yang terhubung dengan *input* yang bernilai paling tinggi akan menerima porsi *weight_delta* (perubahan) yang paling besar. Mengapa? Karena secara matematis, input yang besar memiliki pengaruh yang paling besar terhadap tebakan akhir. Jadi, jika tebakannya salah, input yang besar itulah yang paling "bersalah" dan bobotnya harus dikoreksi paling drastis.
*   **Zero Input = Zero Update:** Jika salah satu *input* bernilai 0, maka `weight_delta` untuk bobot tersebut akan menjadi 0. Ini berarti bobot tersebut tidak akan berubah sama sekali karena ia tidak berkontribusi pada kesalahan tebakan di iterasi tersebut.

### 3. Alpha (Learning Rate) untuk Banyak Bobot
Ketika kita mengoreksi banyak bobot sekaligus, prediksinya menjadi jauh lebih sensitif. Setiap pergeseran kecil pada setiap bobot akan dijumlahkan dan bisa mengubah prediksi secara ekstrem di langkah berikutnya. Oleh karena itu, mengatur *learning rate* (`alpha`) yang tepat menjadi jauh lebih krusial di sini dibandingkan saat hanya menggunakan satu bobot. Jika `alpha` terlalu besar, *error* akan meledak menjadi tak terhingga (*divergence*).

In [ ]:
import numpy as np

# =====================================================================
# REPRODUKSI KODE BAB 5: GRADIENT DESCENT DENGAN MULTIPLE INPUTS
# Mengajarkan jaringan untuk mengoreksi 3 bobot sekaligus
# =====================================================================

# 1. Inisialisasi Pengetahuan Awal (Bobot)
# Kita memiliki 3 bobot untuk 3 input yang berbeda
weights = np.array([0.1, 0.2, -0.1])
alpha = 0.01 # Learning Rate

# 2. Dataset Kita
# (Jumlah Jari, Rasio Menang/Kalah, Jumlah Fans)
toes  = np.array([8.5, 9.5, 9.9, 9.0])
wlrec = np.array([0.65, 0.8, 0.8, 0.9])
nfans = np.array([1.2, 1.3, 0.5, 1.0])

# Mengambil baris pertama dari dataset sebagai observasi uji coba
input_data = np.array([toes[0], wlrec[0], nfans[0]])

# Target Prediksi yang benar (Kebenaran): Tim ini MENANG (1)
goal_pred = 1

def neural_network(input_vector, weight_vector):
    return input_vector.dot(weight_vector)

# 3. Looping Pembelajaran (Training Loop)
print(f"Target Prediksi: {goal_pred}")
print("-" * 50)

for iteration in range(3): # Kita lakukan 3 iterasi saja untuk melihat perubahannya

    # LANGKAH 1: PREDICT (Prediksi dengan 3 input)
    pred = neural_network(input_data, weights)

    # LANGKAH 2: COMPARE (Hitung Error)
    error = (pred - goal_pred) ** 2
    delta = pred - goal_pred

    # LANGKAH 3: LEARN (Menghitung Weight Deltas untuk ke-3 bobot sekaligus)
    # Ini mendistribusikan 'delta' (kesalahan akhir) ke setiap bobot
    # sebanding dengan nilai input masing-masing.
    weight_deltas = input_data * delta

    # LANGKAH 4: UPDATE BOBOT
    weights = weights - (alpha * weight_deltas)

    # Tampilkan hasil
    print(f"Iterasi {iteration+1}")
    print(f"Prediksi: {pred:.4f} | Error: {error:.4f}")
    print(f"Weight Deltas: {weight_deltas}")
    print(f"Bobot Baru   : {weights}\n")

print("Perhatikan bagaimana bobot pertama berubah paling drastis karena input 'toes' (8.5) adalah angka terbesar.")

### Penjelasan Eksekusi Kode:
1.  **Penggunaan NumPy (Skenario Dunia Nyata):** Dalam buku, kode awalnya menggunakan list Python biasa dengan fungsi `for` loop untuk iterasi matematika (disebut `w_sum` dan `ele_mul`). Namun, praktik deep learning yang sesungguhnya menggunakan operasi *Vectorization* NumPy seperti yang kita tulis pada kode di atas `input_data * delta` dan `weights - (alpha * weight_deltas)`. NumPy memproses perhitungan array secara simultan tanpa *looping* manual, yang mempercepat komputasi.
2.  **`weight_deltas = input_data * delta`:** Ini adalah demonstrasi nyata dari *Credit Assignment*. Nilai `delta` (kesalahan) kita adalah satu angka tunggal, tetapi kita kalikan dengan sebuah *array* `input_data` (yang berisi 3 angka). Hasilnya adalah *array* `weight_deltas` yang juga berisi 3 angka.
3.  **Hukuman Proporsional:** Lihat pada output iterasi. Input `toes` (8.5) jauh lebih besar daripada input `wlrec` (0.65). Oleh karena itu, pada `Weight Deltas`, nilai koreksi untuk bobot pertama jauh lebih besar daripada bobot kedua. Jaringan secara otomatis belajar menyalahkan input terbesar atas kesalahan yang terjadi, dan mengubah bobotnya secara drastis untuk kompensasi.